# Estudo de Algoritmos de Recomendação Aplicados a Jogos Digitais

**Trabalho de Conclusão de Curso — Sistemas de Informação**

Este notebook implementa e compara três algoritmos clássicos de recomendação
sobre um dataset público da Steam (Kaggle: *Steam Video Games*).

## Objetivo

Demonstrar, de forma simples e didática, o funcionamento de:

1. **Filtragem colaborativa com similaridade de Pearson** (user-based)
2. **k-Nearest Neighbors com similaridade do cosseno** (item-based)
3. **Recomendação por popularidade** (baseline)

E comparar os três usando métricas básicas: tempo de execução, cobertura
do catálogo e Precision@K (avaliação por holdout).

## Sobre o dataset

O CSV `steam.csv` contém ~200 mil interações no formato:

| Coluna     | Significado                                          |
|------------|------------------------------------------------------|
| `user-id`  | identificador do usuário                             |
| `game`     | nome do jogo                                         |
| `behavior` | `purchase` (comprou) ou `play` (jogou)               |
| `value`    | horas jogadas (quando `behavior=play`)               |

Usaremos as horas jogadas como sinal de interesse.

> **Como usar este notebook**: execute as células em ordem (Shift+Enter).
> Não há dependências entre seções que estejam fora de ordem.


## 1. Imports e configurações

In [1]:
# Bibliotecas usadas em todo o notebook.
# Apenas pandas, numpy, scikit-learn e a biblioteca padrão.
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Sementes fixas garantem que amostras aleatórias sejam reproduzíveis.
np.random.seed(42)

# Configurações de exibição do pandas
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 120)


## 2. Carregando o dataset

In [2]:


# Caminho do Arquivo
CSV_PATH = "steam.csv"

# O CSV original tem 5 colunas (a última é um zero fixo que ignoramos).
df = pd.read_csv(
    CSV_PATH,
    header=0,
    names=["user_id", "game", "behavior", "value", "extra"],
    usecols=["user_id", "game", "behavior", "value"],
)

# Mantém apenas interações de tempo jogado.
df = df[df["behavior"] == "play"].copy()
df = df[["user_id", "game", "value"]]
df.rename(columns={"value": "hours"}, inplace=True)

print(f"Total de interações de play : {len(df):,}")
print(f"Usuários distintos          : {df['user_id'].nunique():,}")
print(f"Jogos distintos             : {df['game'].nunique():,}")
df.head()


Total de interações de play : 70,489
Usuários distintos          : 11,350
Jogos distintos             : 3,600


,user_id,game,hours
1,151603712,The Elder Scrolls V Skyrim,273.0
3,151603712,Fallout 4,87.0
5,151603712,Spore,14.9
7,151603712,Fallout New Vegas,12.1
9,151603712,Left 4 Dead 2,8.9


## 3. Filtrando usuários e jogos pouco representativos

In [3]:
MIN_GAMES_PER_USER = 5
MIN_USERS_PER_GAME = 5

for _ in range(2):  # aplica em duas passadas (efeito cascata)
    n_games_by_user = df.groupby("user_id")["game"].count()
    valid_users = n_games_by_user[n_games_by_user >= MIN_GAMES_PER_USER].index
    df = df[df["user_id"].isin(valid_users)]

    n_users_by_game = df.groupby("game")["user_id"].count()
    valid_games = n_users_by_game[n_users_by_game >= MIN_USERS_PER_GAME].index
    df = df[df["game"].isin(valid_games)]

print(f"Após filtragem:")
print(f"  Usuários : {df['user_id'].nunique():,}")
print(f"  Jogos    : {df['game'].nunique():,}")
print(f"  Linhas   : {len(df):,}")


Após filtragem:
  Usuários : 2,387
  Jogos    : 1,515
  Linhas   : 53,798


## 4. Construindo a matriz usuário × jogo

In [4]:
# Transformação log para suavizar outliers.
df["rating"] = np.log1p(df["hours"])

# Pivota para o formato usuário × jogo.
matrix = df.pivot_table(
    index="user_id",
    columns="game",
    values="rating",
    aggfunc="mean",
)

print(f"Forma da matriz : {matrix.shape}  (usuários x jogos)")
sparsity = matrix.isna().sum().sum() / matrix.size
print(f"Esparsidade     : {sparsity * 100:.2f}%")


Forma da matriz : (2387, 1515)  (usuários x jogos)
Esparsidade     : 98.51%


## 5. Estatísticas básicas e jogos mais populares

In [5]:
# Top-10 jogos por número de usuários distintos.
top_games = (
    df.groupby("game")["user_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)
print("Top 10 jogos por nº de usuários:\n")
print(top_games.to_string())


Top 10 jogos por nº de usuários:

game
Team Fortress 2                    1175
Counter-Strike Global Offensive     932
Dota 2                              918
Left 4 Dead 2                       663
Unturned                            645
Garry's Mod                         585
The Elder Scrolls V Skyrim          562
Counter-Strike Source               448
Portal 2                            422
Terraria                            407


## 6. Algoritmo 1 — Filtragem Colaborativa com Pearson

In [6]:


def recommend_pearson(user_id, matrix, n=5, k_neighbors=40, min_support=3):

    target = matrix.loc[user_id]
    user_means = matrix.mean(axis=1)          # média de cada usuário (ignora NaN)
    target_mean = user_means[user_id]

    # Suprime avisos do numpy quando dois usuários não compartilham jogos
    # (correlação fica indefinida — descartada logo a seguir).
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        # corrwith compara cada usuário (linha) com o usuário-alvo.
        sims = matrix.T.corrwith(target)

    sims = sims.drop(index=user_id, errors="ignore").dropna()

    # Mantém apenas vizinhos com correlação positiva (gostos parecidos).
    sims = sims[sims > 0].nlargest(k_neighbors)
    if sims.empty:
        return pd.Series(dtype=float)

    # Notas dos vizinhos, centradas na média de cada um (r_vi - r̄_v).
    neighbors = matrix.loc[sims.index]
    centered = neighbors.sub(user_means[sims.index], axis=0)
    rated = neighbors.notna()                 # onde o vizinho realmente jogou

    weights = sims.values.reshape(-1, 1)

    # Predição = média do usuário-alvo + média ponderada dos desvios dos vizinhos.
    weighted_sum = (centered.fillna(0).values * weights).sum(axis=0)
    sim_sum = (np.abs(weights) * rated.values).sum(axis=0)
    sim_sum[sim_sum == 0] = 1e-9              # evita divisão por zero
    predicted = target_mean + weighted_sum / sim_sum

    # Quantos vizinhos sustentam cada item (suporte).
    support = rated.values.sum(axis=0)

    scores = pd.Series(predicted, index=matrix.columns)
    support = pd.Series(support, index=matrix.columns)

    # Remove jogos já jogados pelo usuário-alvo.
    already_played = target.dropna().index
    scores = scores.drop(labels=already_played, errors="ignore")
    support = support.reindex(scores.index)

    # Significance weighting: só recomenda itens com suporte mínimo de vizinhos.
    scores = scores[support >= min_support]

    return scores.sort_values(ascending=False).head(n)


In [7]:
# Escolhe um usuário com bastante histórico para o exemplo correr bem.
sample_user = matrix.notna().sum(axis=1).sort_values(ascending=False).index[5]
print(f"Usuário escolhido : {sample_user}")
print(f"Nº de jogos dele  : {matrix.loc[sample_user].notna().sum()}")
print()
print("Top 5 jogos dele (por horas):")
print(df[df["user_id"] == sample_user]
      .nlargest(5, "hours")[["game", "hours"]]
      .to_string(index=False))


Usuário escolhido : 48798067
Nº de jogos dele  : 225

Top 5 jogos dele (por horas):
                           game  hours
          Mount & Blade Warband 3178.0
Counter-Strike Global Offensive 1302.0
          Counter-Strike Source  795.0
                    War Thunder  249.0
             Napoleon Total War  174.0


In [8]:
print("Recomendações com PEARSON:\n")
recs_pearson = recommend_pearson(sample_user, matrix, n=10)
print(recs_pearson.to_string())


Recomendações com PEARSON:

game
Sid Meier's Civilization V                     4.013320
Call of Duty Modern Warfare 3 - Multiplayer    3.123383
Counter-Strike                                 3.065464
Plants vs. Zombies Game of the Year            2.947322
Football Manager 2011                          2.943305
Counter-Strike Condition Zero                  2.384554
Call of Duty Ghosts                            2.303378
Call of Duty Modern Warfare 3                  1.893887
Call of Duty Ghosts - Multiplayer              1.394974
Day of Defeat                                  1.042975


## 7. Algoritmo 2 — k-Nearest Neighbors (item-based, cosseno)


In [9]:
# "Treinar" o k-NN uma vez só sobre a matriz transposta (jogos como linhas).
# Cada jogo vira um vetor com a "assinatura" de quem o jogou.
item_user_matrix = matrix.T.fillna(0).values
game_names = matrix.columns

knn_model = NearestNeighbors(metric="cosine", algorithm="brute")
knn_model.fit(item_user_matrix)

print(f"Modelo k-NN pronto: {item_user_matrix.shape[0]} jogos indexados.")


Modelo k-NN pronto: 1515 jogos indexados.


In [10]:


def recommend_knn(user_id, matrix, n=5, k_neighbors=10):
    user_row = matrix.loc[user_id]
    played = user_row.dropna()
    if played.empty:
        return pd.Series(dtype=float)

    scores = pd.Series(0.0, index=game_names)

    # Para cada jogo que o usuário jogou, busca jogos parecidos.
    # Pedimos k+1 porque o jogo é vizinho de si mesmo (distância 0).
    for game, rating in played.items():
        idx = game_names.get_loc(game)
        distances, indices = knn_model.kneighbors(
            item_user_matrix[idx].reshape(1, -1),
            n_neighbors=k_neighbors + 1,
        )
        # Distância cosseno -> similaridade cosseno
        sims = 1 - distances.flatten()
        neighbor_games = game_names[indices.flatten()]

        for neighbor, sim in zip(neighbor_games, sims):
            if neighbor == game:
                continue  # ignora o próprio jogo
            # Acumula: rating do jogo conhecido * similaridade do vizinho
            scores[neighbor] += rating * sim

    # Remove jogos já jogados.
    scores = scores.drop(labels=played.index, errors="ignore")
    return scores[scores > 0].sort_values(ascending=False).head(n)


In [11]:
print("Recomendações com k-NN:\n")
recs_knn = recommend_knn(sample_user, matrix, n=5)
print(recs_knn.to_string())


Recomendações com k-NN:

game
Portal 2                             19.932586
Mount & Blade With Fire and Sword    13.096504
Total War ATTILA                      9.881593
Left 4 Dead                           9.095099
Medieval II Total War Kingdoms        7.893027


## 8. Algoritmo 3 — Popularidade (baseline)

In [12]:
# Pré-calcula uma vez: número de usuários distintos por jogo.
# .notna() é True onde o usuário jogou; .sum() conta os True por coluna.
popularity = matrix.notna().sum().sort_values(ascending=False)


def recommend_popularity(user_id, matrix, n=5):
    """Retorna os N jogos mais populares que o usuário ainda não jogou."""
    already_played = matrix.loc[user_id].dropna().index
    return (
        popularity
        .drop(labels=already_played, errors="ignore")
        .head(n)
        .astype(float)
    )


print("Recomendações por POPULARIDADE:\n")
recs_pop = recommend_popularity(sample_user, matrix, n=5)
print(recs_pop.to_string())


Recomendações por POPULARIDADE:

game
Portal 2                      422.0
Sid Meier's Civilization V    386.0
Warframe                      365.0
Robocraft                     315.0
Counter-Strike                280.0


## 9. Comparando os três algoritmos lado a lado

In [13]:
recs_pearson = recommend_pearson(sample_user, matrix, n=5)
recs_knn = recommend_knn(sample_user, matrix, n=5)
recs_pop = recommend_popularity(sample_user, matrix, n=5)

comparison = pd.DataFrame({
    "Pearson"    : recs_pearson.index.tolist()    + [""] * (5 - len(recs_pearson)),
    "k-NN"       : recs_knn.index.tolist()        + [""] * (5 - len(recs_knn)),
    "Popularidade": recs_pop.index.tolist()       + [""] * (5 - len(recs_pop)),
}, index=[f"Top {i}" for i in range(1, 6)])

print(f"Recomendações para o usuário {sample_user}:\n")
comparison


Recomendações para o usuário 48798067:



,Pearson,k-NN,Popularidade
Top 1,Sid Meier's Civilization V,Portal 2,Portal 2
Top 2,Call of Duty Modern Warfare 3 - Multiplayer,Mount & Blade With Fire and Sword,Sid Meier's Civilization V
Top 3,Counter-Strike,Total War ATTILA,Warframe
Top 4,Plants vs. Zombies Game of the Year,Left 4 Dead,Robocraft
Top 5,Football Manager 2011,Medieval II Total War Kingdoms,Counter-Strike


In [14]:
# Quantas recomendações cada par tem em comum?
sets = {
    "Pearson"     : set(recs_pearson.index),
    "k-NN"        : set(recs_knn.index),
    "Popularidade": set(recs_pop.index),
}

print("Concordância (jogos em comum entre os Top-5):")
print(f"  Pearson  ∩ k-NN          : {len(sets['Pearson'] & sets['k-NN'])}")
print(f"  Pearson  ∩ Popularidade  : {len(sets['Pearson'] & sets['Popularidade'])}")
print(f"  k-NN     ∩ Popularidade  : {len(sets['k-NN'] & sets['Popularidade'])}")
print(f"  Comum aos três          : {len(sets['Pearson'] & sets['k-NN'] & sets['Popularidade'])}")


Concordância (jogos em comum entre os Top-5):
  Pearson  ∩ k-NN          : 0
  Pearson  ∩ Popularidade  : 2
  k-NN     ∩ Popularidade  : 1
  Comum aos três          : 0


## 10. Métricas — Tempo de execução

Medimos quantos milissegundos cada algoritmo gasta em **uma chamada**
de recomendação (5 jogos para 1 usuário).


In [15]:
def measure_time(func, *args, **kwargs):
    """Roda a função uma vez e retorna o tempo em milissegundos."""
    start = time.perf_counter()
    func(*args, **kwargs)
    return (time.perf_counter() - start) * 1000


times = {
    "Pearson"     : measure_time(recommend_pearson, sample_user, matrix, n=5),
    "k-NN"        : measure_time(recommend_knn, sample_user, matrix, n=5),
    "Popularidade": measure_time(recommend_popularity, sample_user, matrix, n=5),
}

times_df = pd.DataFrame(times.items(), columns=["Algoritmo", "Tempo (ms)"])
times_df["Tempo (ms)"] = times_df["Tempo (ms)"].round(2)
times_df


,Algoritmo,Tempo (ms)
0,Pearson,396.87
1,k-NN,4326.03
2,Popularidade,0.49


## 11. Métricas — Cobertura do catálogo


In [16]:
# Amostra de usuários para o cálculo (precisa de mais de um!)
SAMPLE_SIZE = 30
sample_users = np.random.choice(matrix.index, size=SAMPLE_SIZE, replace=False)

algos = {
    "Pearson"     : recommend_pearson,
    "k-NN"        : recommend_knn,
    "Popularidade": recommend_popularity,
}

coverage = {}
total_games = matrix.shape[1]

for name, func in algos.items():
    games_recommended = set()
    for u in sample_users:
        recs = func(u, matrix, n=5)
        games_recommended.update(recs.index)
    coverage[name] = {
        "Jogos únicos recomendados": len(games_recommended),
        "Cobertura (%)": round(100 * len(games_recommended) / total_games, 2),
    }

coverage_df = pd.DataFrame(coverage).T
print(f"Cobertura sobre {SAMPLE_SIZE} usuários, Top-5 cada:")
coverage_df


Cobertura sobre 30 usuários, Top-5 cada:


,Jogos únicos recomendados,Cobertura (%)
Pearson,65.0,4.29
k-NN,67.0,4.42
Popularidade,13.0,0.86


## 12. Métricas — Precision@K (avaliação por holdout)

### O protocolo

1. Para cada usuário da amostra, escolhemos **2 jogos que ele realmente jogou**
   e os **escondemos** da matriz (substituímos por `NaN`).
2. Pedimos ao algoritmo as **Top-5** recomendações para esse usuário.
3. **Precision@5** = (jogos escondidos que apareceram no Top-5) / 5.
4. Tiramos a média entre todos os usuários da amostra.

In [17]:
def precision_at_k(func, sample_users, matrix, k=5, hold_out=2):

    global knn_model, item_user_matrix, popularity, game_names

    precisions = []

    for u in sample_users:
        user_games = matrix.loc[u].dropna()
        if len(user_games) <= hold_out:
            continue

        # Esconde 'hold_out' jogos do usuário.
        hidden = list(user_games.sample(hold_out, random_state=42).index)
        modified = matrix.copy()
        modified.loc[u, hidden] = np.nan

        # Se for k-NN, precisamos reconstruir o índice item-usuário.
        if func is recommend_knn:
            saved_knn, saved_items, saved_names = knn_model, item_user_matrix, game_names
            item_user_matrix = modified.T.fillna(0).values
            game_names = modified.columns
            knn_model = NearestNeighbors(metric="cosine", algorithm="brute")
            knn_model.fit(item_user_matrix)

        # Se for Popularidade, recalculamos sobre a matriz modificada.
        if func is recommend_popularity:
            saved_pop = popularity
            popularity = modified.notna().sum().sort_values(ascending=False)

        recs = func(u, modified, n=k)
        hits = len(set(recs.index) & set(hidden))
        precisions.append(hits / k)

        # Restaura estado global.
        if func is recommend_knn:
            knn_model, item_user_matrix, game_names = saved_knn, saved_items, saved_names
        if func is recommend_popularity:
            popularity = saved_pop

    return float(np.mean(precisions)) if precisions else 0.0


In [18]:
# Roda Precision@K para os três algoritmos.
# Pode levar alguns segundos (especialmente o k-NN).
print(f"Calculando Precision@5 sobre {len(sample_users)} usuários...\n")

precision_results = {}
for name, func in algos.items():
    start = time.perf_counter()
    p = precision_at_k(func, sample_users, matrix, k=5, hold_out=2)
    elapsed = time.perf_counter() - start
    precision_results[name] = round(p, 4)
    print(f"  {name:<13} : Precision@5 = {p:.4f}   (calculado em {elapsed:.1f}s)")


Calculando Precision@5 sobre 30 usuários...

  Pearson       : Precision@5 = 0.0133   (calculado em 9.5s)
  k-NN          : Precision@5 = 0.1067   (calculado em 14.6s)
  Popularidade  : Precision@5 = 0.1000   (calculado em 0.4s)


## 13. Relatório consolidado

In [21]:
# Junta tudo em uma única tabela.
report = pd.DataFrame({
    "Algoritmo"   : list(algos.keys()),
    "Tempo (ms)"  : [round(times[a], 2) for a in algos],
    "Cobertura (%)" : [coverage[a]["Cobertura (%)"] for a in algos],
    "Jogos únicos": [coverage[a]["Jogos únicos recomendados"] for a in algos],
    "Precision@5" : [precision_results[a] for a in algos],
})

print("RELATÓRIO FINAL\n")
report


RELATÓRIO FINAL



,Algoritmo,Tempo (ms),Cobertura (%),Jogos únicos,Precision@5
0,Pearson,468.36,4.29,65,0.03
1,k-NN,5248.65,3.37,51,0.11
2,Popularidade,0.60,0.86,13,0.10
